In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [4]:
df = pd.read_csv('данные.csv')
df['date'] = pd.to_datetime(df['date'])
df.head()

,date,user_id,view_adverts
0,2023-11-11,8c020470-8461-11ed-83d0-552e8cc749d6,13
1,2023-11-18,5875f070-7b92-11ee-a6fb-8b298e83f4f7,14
2,2023-11-29,3c2d27c0-4fd6-11eb-b89f-2ffb31b67dd6,21
3,2023-11-29,234a96d0-ad16-11ed-a2e6-793ddfeeba1f,23
4,2023-11-29,4d07c180-644f-11eb-879c-b7c02edf4f37,12


## Вопрос 1: MAU

In [5]:
mau = df['user_id'].nunique()
print(f'MAU = {mau}')

MAU = 7639


## Вопрос 2: DAU

In [6]:
dau_per_day = df.groupby('date')['user_id'].nunique()
dau = round(dau_per_day.mean())
print(f'DAU (среднее) = {dau}')
print(f'DAU по дням:\n{dau_per_day}')

DAU (среднее) = 560
DAU по дням:
date
2023-11-01    623
2023-11-02    649
2023-11-03    573
2023-11-04    343
2023-11-05    350
2023-11-06    660
2023-11-07    629
2023-11-08    600
2023-11-09    661
2023-11-10    583
2023-11-11    354
2023-11-12    377
2023-11-13    646
2023-11-14    687
2023-11-15    690
2023-11-16    639
2023-11-17    585
2023-11-18    412
2023-11-19    378
2023-11-20    711
2023-11-21    651
2023-11-22    608
2023-11-23    632
2023-11-24    595
2023-11-25    366
2023-11-26    372
2023-11-27    589
2023-11-28    610
2023-11-29    621
2023-11-30    620
Name: user_id, dtype: int64


## Вопрос 3: Retention Day 1 (пользователи 1 ноября)

In [7]:
nov1_users = set(df[df['date'] == '2023-11-01']['user_id'])
nov2_users = set(df[df['date'] == '2023-11-02']['user_id'])
retained = nov1_users & nov2_users

retention_d1 = len(retained) / len(nov1_users) * 100
print(f'Пользователей 1 ноября: {len(nov1_users)}')
print(f'Вернулись 2 ноября: {len(retained)}')
print(f'Retention Day 1 = {retention_d1:.1f}%')

Пользователей 1 ноября: 623
Вернулись 2 ноября: 166
Retention Day 1 = 26.6%


## Вопрос 5: Конверсия в просмотр объявления

In [9]:
users_with_views = df[df['view_adverts'] > 0]['user_id'].nunique()
total_users = df['user_id'].nunique()
conversion = users_with_views / total_users * 100

print(f'Пользователи с просмотрами > 0: {users_with_views}')
print(f'Всего уникальных пользователей: {total_users}')
print(f'Конверсия = {conversion:.1f}%')

Пользователи с просмотрами > 0: 3538
Всего уникальных пользователей: 7639
Конверсия = 46.3%


## Вопрос 6: Среднее кол-во просмотренных объявлений на пользователя

In [12]:
total_views = df['view_adverts'].sum()
unique_users = df['user_id'].nunique()
avg_views = total_views / unique_users

print(f'Всего просмотров: {total_views}')
print(f'Уникальных пользователей: {unique_users}')
print(f'Среднее на пользователя = {avg_views:.1f}')

Всего просмотров: 21914
Уникальных пользователей: 7639
Среднее на пользователя = 2.9


## Вопрос 7: NPS

In [15]:
total = 2000
promoters = 1200
detractors = 500
neutrals = 300

nps = (promoters / total - detractors / total) * 100
print(f'% сторонников = {promoters/total*100}%')
print(f'% критиков = {detractors/total*100}%')
print(f'NPS = {nps:.0f}%')

% сторонников = 60.0%
% критиков = 25.0%
NPS = 35%


## Вопрос 8: A/B тесты ARPU

In [22]:
ab = pd.read_csv('аб.csv')

for exp in [1, 2, 3]:
    data = ab[ab['experiment_num'] == exp]
    test = data[data['experiment_group'] == 'test']['revenue']
    control = data[data['experiment_group'] == 'control']['revenue']
    
    t_stat, p_tt = stats.ttest_ind(test, control)
    u_stat, p_mw = stats.mannwhitneyu(test, control, alternative='two-sided')
    
    print(f'Эксперимент {exp}:')
    print(f'  ARPU test = {test.mean():.2f}, ARPU control = {control.mean():.2f}')
    print(f'  t-test p-value = {p_tt:.6f}')
    print(f'  Mann-Whitney p-value = {p_mw:.6f}')
    print(f'  Значимо (α=0.05): t-test={p_tt < 0.05}, MW={p_mw < 0.05}')
    print()

Эксперимент 1:
  ARPU test = 665.74, ARPU control = 722.46
  t-test p-value = 0.688784
  Mann-Whitney p-value = 0.796006
  Значимо (α=0.05): t-test=False, MW=False

Эксперимент 2:
  ARPU test = 332.93, ARPU control = 704.65
  t-test p-value = 0.000992
  Mann-Whitney p-value = 0.008453
  Значимо (α=0.05): t-test=True, MW=True

Эксперимент 3:
  ARPU test = 998.67, ARPU control = 663.21
  t-test p-value = 0.061743
  Mann-Whitney p-value = 0.001001
  Значимо (α=0.05): t-test=False, MW=True



## Вопрос 9: Средний доход на пользователя

In [23]:
listers = pd.read_csv('листеры.csv')

revenue_per_user = listers.groupby('user_id')['revenue'].sum()
mean_revenue = revenue_per_user.mean()

print(f'Пользователей: {len(revenue_per_user)}')
print(f'Средний доход на пользователя: {mean_revenue:.1f}')

Пользователей: 31
Средний доход на пользователя: 156.5


## Вопрос 10: Медиана возраста

In [24]:
age_per_user = listers.groupby('user_id')['age'].first()
median_age = age_per_user.median()

print(f'Медиана возраста: {median_age}')

Медиана возраста: 28.0


## Вопрос 18: A/B тест (конверсия в платёж)

In [27]:
n_a, conv_a = 100047501, 1003
n_b, conv_b = 100001055, 1099

p_a = conv_a / n_a
p_b = conv_b / n_b

p_pool = (conv_a + conv_b) / (n_a + n_b)
se = (p_pool * (1 - p_pool) * (1/n_a + 1/n_b)) ** 0.5
z = (p_b - p_a) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print(f'CR_A = {p_a*100:.6f}%')
print(f'CR_B = {p_b*100:.6f}%')
print(f'Z = {z:.4f}, p-value = {p_value:.6f}')
print(f'Lift = {(p_b - p_a) / p_a * 100:.2f}%')
print(f'Статистически значимо: {p_value < 0.05}')

CR_A = 0.001003%
CR_B = 0.001099%
Z = 2.1046, p-value = 0.035330
Lift = 9.62%
Статистически значимо: True
